# GLM-5.3-Flash on 4x CMP 170HX (SM80) — 4card-tp4, vLLM

| Metric | Value |
|---|---|
| Decode, c=1 (median of 5 reps) | 56.4 tok/s (peak 56.9) |
| Best aggregate | 37.0 tok/s at c=8 (below the EXL3 baseline; TP4 all-reduce bound) |
| Prefill | untested |
| TTFT | untested |

__omp_shell("[sweep chart](../assets/charts/2026-09-03-glm-5.3-flash-vllm-sm80-4gpu-sweep.png)")

```bash
docker pull ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-20260903
```

Guide: [docs/models/glm-5.3-flash.md](../docs/models/glm-5.3-flash.md) — this notebook is the executed proof of the 2026-09-03 run.

In [ ]:
# --- Status cell ---
EXPERIMENT = "2026-09-03-glm-5.3-flash-vllm-sm80-4gpu"
RESULTS_DIR = "../results/" + EXPERIMENT
LIVE = False
print(f"experiment : {EXPERIMENT}")
print(f"LIVE       : {LIVE}")
print("status     : measured (TP4 + MTP-3 first boot + depth sweep; aggregate below EXL3, PP4+MTP port pending)")

In [ ]:
# --- Helper: table renderer ---
from IPython.display import display, Markdown

def render_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |",
             "|" + "|".join(["---"] * len(headers)) + "|"]
    for row in rows:
        lines.append("| " + " | ".join(str(c) for c in row) + " |")
    display(Markdown("\n".join(lines)))

## 1. Identity and pins

GLM-5.3-Flash served by the club's `sm80vllm` fork, branch `glm53-sm80`
(wtdcode's GLM SM80 enablement vendored with attribution; provenance in
that branch's `docs/SM80.md`). This is the first working vLLM lane for
this model on CMP 170HX — it resolves the 2026-08-31 compatibility
review's "every vLLM-served checkpoint blocked on SM80" finding.

In [ ]:
pins = {
    "model": "GLM-5.3-Flash",
    "checkpoint": "wtdcode/GLM-5.3-Flash-AWQ-W4A16",
    "checkpoint_revision": "abd7b07719111f137e1de8a0c1b7e01c11b74d1a",
    "checkpoint_bytes": 190843146533,
    "quantization": "AWQ W4A16 (compressed-tensors)",
    "fork_branch": "glm53-sm80 @ f6fbf3b854 (PixelML/sm80vllm)",
    "image": "glm53-sm80:test (f7fe5d02c295) -> ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-20260903",
    "driver": "610.43.03",
    "topology": "TP4, 4x CMP 170HX, 180 W per card",
    "speculative": "native MTP, num_speculative_tokens=3 (swept 2/3/5)",
}
for k, v in pins.items():
    print(f"{k:20}: {v}")

## 2. Preflight and safety

Forced airflow confirmed before the run; 180 W per-card power cap
verified (`nvidia-smi --query-gpu=index,power.limit` shows 180.00 W on
all four); no Xid or ECC events at any point during the boots, sweeps,
or the crash-recovery cycles documented in the appendix.

In [ ]:
preflight = {
    "forced_airflow_confirmed": True,
    "power_limit_W_per_card": 180,
    "xid_ecc_events": 0,
    "peak_temp_C": None,  # not recorded this run; watch noted clean on sibling EXL3 run
}
preflight

## 2. Visible results

### MTP depth sweep, c=1 decode (5 measured reps each, temperature 0.7, 512 output tokens, `ignore_eos`; first rep treated as cold)

k=3 is the optimum — the same acceptance-cliff shape as the DSpark
k-sweep in `docs/LESSONS.md` §d. Deeper drafts rerun the single MTP
layer more times without extending acceptance.

In [ ]:
sweep = [
    ("k=2", 51.1, 54.7, 38.7),
    ("k=3", 56.4, 56.9, 41.0),
    ("k=5", 47.1, 52.3, 35.0),
]
render_table(["num_speculative_tokens", "median tok/s", "peak", "cold rep"], sweep)

### Aggregate vs the EXL3 baseline (same boot)

c=1 doubles the EXL3 lane. c=8 aggregate **loses** to EXL3: TP4 runs an
all-reduce per layer across PCIe Gen1 with no P2P. The playbook fix is
PP4 + MTP (PP moves ~28x less wire data on this fabric, per
`docs/LESSONS.md` §c), which is stock-blocked in vLLM and needs the
MTP-under-PP patch set (upstream PR #46994) ported — see appendix.

In [ ]:
agg = [
    ("c=1", 54.7, 25.2),
    ("c=8", 37.0, 44.8),
]
render_table(["Concurrency", "vLLM sm80 TP4+MTP-3 (aggregate tok/s)", "EXL3 4.05bpw baseline (180 W)"], agg)
display(Markdown(
    "![sweep chart](../assets/charts/2026-09-03-glm-5.3-flash-vllm-sm80-4gpu-sweep.png)"))

## 3. Reproduce

**Hardware.** 4x NVIDIA CMP 170HX (SM80, 64 GiB HBM2e each, no NVLink,
no P2P over PCIe), 180 W per-card power limit, forced airflow.

**Weights.** Staged copy on local NVMe is strongly recommended — the
190.8 GB checkpoint streams at 54 s/shard warm from disk versus a
~41-minute load from NFS.

### Download and stage the weights

```bash
pip install -U huggingface_hub
hf download wtdcode/GLM-5.3-Flash-AWQ-W4A16 \
  --revision abd7b07719111f137e1de8a0c1b7e01c11b74d1a \
  --local-dir <weights>
```
Verify: 24 files, 190,843,146,533 bytes total, 0 missing/mismatched.

### Launch

```bash
docker pull ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-20260903

docker run -d --name glm53-vllm --gpus '"device=0,1,2,3"' \
  --shm-size 16g --ipc=host -p 127.0.0.1:18098:8000 \
  -e HF_HUB_OFFLINE=1 -e VLLM_WORKER_MULTIPROC_METHOD=spawn \
  -e VLLM_ENGINE_READY_TIMEOUT_S=1800 \
  -e VLLM_ENGINE_ITERATION_TIMEOUT_S=1800 \
  -e NCCL_VERSION=2.28.3-1 -e TORCH_CUDA_ARCH_LIST=8.0 \
  -e VLLM_TARGET_DEVICE=cuda \
  --mount type=bind,src=<weights>,dst=/model,readonly \
  ghcr.io/pixelml/club-170hx:vllm-glm53-sm80-20260903 \
  vllm serve /model --served-model-name glm-5.3-flash \
  --tensor-parallel-size 4 --max-model-len 524288 \
  --gpu-memory-utilization 0.92 --max-num-seqs 16 \
  --max-num-batched-tokens 8192 --enable-prefix-caching \
  --disable-custom-all-reduce \
  --compilation-config '{"cudagraph_mode":"FULL_AND_PIECEWISE","cudagraph_capture_sizes":[1,2,4,8,16],"max_cudagraph_capture_size":16}' \
  --speculative-config '{"method":"mtp","num_speculative_tokens":3}' \
  --enable-auto-tool-choice --tool-call-parser glm47 --reasoning-parser glm45
```

**Do not add `--no-enable-flashinfer-autotune` with MTP** — it crashes
at engine startup and wedges a GPU at the PCIe level (appendix).

**Expected boot time.** Weights load 54 s/shard warm from NVMe
(~10 min total including engine init and CUDA graph capture); the very
first run on cold page cache is slower.

**Bench.** OpenAI-compatible chat completions on
`http://127.0.0.1:18098/v1`; count tokens from the final usage object;
5 reps, discard the cold rep from the headline.

## 4. Appendix

<details>
<summary>Failure history, the autotune hazard, limitations (click to expand)</summary>

### Boot-path failures fixed on the way to the first boot
1. **Missing vendored patch** — the first `glm53-sm80:test` build failed
   at `COPY patches/safetensors_torch_f8_e8m0.py`. Fix: vendored the
   overlay from the `consolidate-sm80` branch (commit `623f59ba65`).
2. **`rustc` off PATH** — build attempt 2 failed at `RUN rustc --version`
   (exit 127) because an `ENV CARGO_HOME` override moved rustup binaries
   off PATH. Fix: dropped the override (`f6fbf3b854`).
3. **Driver-less import gate** — the in-build gate importing
   `Glm5NextForCausalLM` needs a GPU driver (fp8_sm80.py touches
   `tl.constexpr` at import time). Fix: scope the in-build gate to
   driver-free checks; run the GPU-dependent imports as a post-build
   `docker run --gpus` gate.

### The `--no-enable-flashinfer-autotune` + MTP-3 hazard
That flag combination crashes the engine at startup
(`cudaErrorLaunchFailure` in mm-encoder profiling), **reproduced on a
pristine post-reboot boot**, and each crash wedges a GPU at the PCIe
level (`rev ff` in `lspci`); recovery required a VM reboot — a guest-side
PCI rescan and driver module reload were not enough for the
fallen-off device. The measured recipe keeps flashinfer autotune at its
default (on). This hazard is MTP-3-specific: MTP-2 and MTP-5 boots with
the same flag served fine.

### PP4 + MTP is the open aggregate item
PP4 boots without MTP pass; every PP4 + MTP attempt OOM'd — the last PP
rank carries the MTP draft and the LM head on top of its stage, and the
draft-logits spike (batched tokens x draft tokens x 154,880 vocab) blew
the budget (4.10 GiB single allocation observed). The upstream fix is
PR #46994 (MTP under PP on the V2 model runner, which this fork already
uses); it does not apply cleanly to this base (different upstream era)
and is not ported yet. Community reference recipe on a GLM-5.2 quant:
`--enforce-eager`, gpu-memory-utilization 0.80.

### Other limitations
- fp8 KV cache and block-size 256 are rejected by the Triton MLA sparse
  backend on this model (attention-selector ValueError) — the DSV4
  playbook pins do not transfer here, and per `docs/LESSONS.md` fp8 KV
  costs decode on this hardware anyway.
- Prefill and TTFT were not measured this run.
- c=1 was measured with temperature 0.7 (`ignore_eos`), not the greedy
  protocol of the sibling EXL3 receipts; medians are stable across reps
  but the protocols differ between lanes.

### Links
- Live evidence thread: [seanphan/pixelml#103](https://github.com/seanphan/pixelml/issues/103)
  (first boot: comment 5527939703; depth sweep: comment 5530227651).
- Raw receipts: [results/2026-09-03-glm-5.3-flash-vllm-sm80-4gpu](../results/2026-09-03-glm-5.3-flash-vllm-sm80-4gpu/README.md).
- Fork branch: [PixelML/sm80vllm `glm53-sm80`](https://github.com/PixelML/sm80vllm/tree/glm53-sm80).

</details>